## EDA – Analyse exploratoire du dataset "CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026"

### Contexte

- Projet de création d'un CRM sur CremeCRM.
- Réception d'un premier fichier de données contenant des informations de contacts (entités, individus, adresses e-mail),
- ces données constituent une première extraction et seront enrichiers au fil du temps.

Avant toute intégration dans le CRM, une **EDA (analyse exploratoire des données)** est réalisée afin de :
- vérifier la structure des données,
- identifier d’éventuels problèmes de qualité,
- préparer les règles nécessaires à un futur ETL.

### Objectifs de ce notebook

Ce notebook a pour objectifs de :
- comprendre la structure et le contenu des données reçues,
- détecter les incohérences, valeurs manquantes ou formats hétérogènes,
- formuler des constats utiles pour préparer un ETL fiable.

### Périmètre

- Ce notebook se concentre uniquement sur l’EDA.
- Aucune transformation définitive ni import dans le CRM n’est réalisé à ce stade.
- Les données sources brutes sont stockées dans `data/raw/` et ne sont pas versionnées.
- Le notebook est versionné afin de conserver la traçabilité de l’analyse.

### Description du fichier source

Informations générales (infos visibles sans code)

- Nom du fichier :  
  `CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xlsx`
- Format : Excel
- Date de réception : janvier 2026
- Nombre de feuilles : 3

Description des feuilles :

Feuille 1 — CONTACTS DIRECTEUR-DRH-SERVICE

Cette feuille contient des informations relatives :
- aux entités (entreprises, organisations),
- aux individus associés (directeurs, DRH, responsables RH, tuteurs),
- aux adresses postales,
- aux coordonnées de contact (notamment e-mail, lorsqu’il est renseigné).

Volumétrie observée/visible :
- lignes : 6 799
- colonnes : 14

Premières observations générales :
- présence de champs vides,
- emails manquants,
- hétérogénéité des formats (espaces, majuscules, accents).

Les colonnes peuvent être regroupées par thématique :

**Identité entité**
- `Code.Entité` : identifiant / nom de l'entreprise
- `Libellé.Entité` : nom de l'entreprise
- `Code.Type d'entité` : type de structure
- `Assujetti.Entité` : assujettissement à la taxe d'apprentissage (vrai ou vide)

**Adresse**
- `Rue (ligne 1).Adresse` : adresse principale
- `Rue (ligne 2).Adresse` à `Rue (ligne 4).Adresse` : compléments d’adresse
- `Code postal.Ville` : code postal
- `Nom.Ville` : nom de la ville

**Contact**
- `Libellé.Titre` : civilité (Madame / Monsieur)
- `Nom.Individu` : nom de famille
- `Prénom.Individu` : prénom
- `Coordonnée.Coordonnée` : information de contact

La colonne `Coordonnée.Coordonnée` contient des valeurs hétérogènes :
- adresses e-mail professionnelles,
- mentions textuelles (ex. : *PAS D’EMAILING*, *DESINSCRIPTION E-MAILING*, *NC*),
- parfois des numéros de téléphone.

Ces éléments indiquent que ce champ nécessitera un traitement spécifique lors du nettoyage des données.

Feuille 2 — Taxe

Cette feuille contient une liste d’adresses e-mail correspondant aux verseurs de taxe 2025.

- Structure : une seule colonne
- Volumétrie observée/visible : 101 lignes
- Contenu : adresses e-mail uniquement

Feuille 3 — Pour e-mailing

Cette feuille contient une base d’adresses e-mail destinée à des actions d’e-mailing.

- Structure : une seule colonne (`Coordonnee.Coordonnee`)
- Volumétrie observée/visible : 5 563 lignes
- Contenu : adresses e-mail uniquement


# EDA – Feuille 1 : CONTACTS DIRECTEUR-DRH-SERVICE

### Objectif :
- analyser la structure et le contenu de la feuille "CONTACTS DIRECTEUR-DRH-SERVICE",
- évaluer la qualité des données (doublons, valeurs manquantes et formats hétérogènes) et leur exploitabilité,
- identifier les points à traiter lors de la phase ETL,
- générer un JSONL intermédiaire.


## 1. Imports et chargement des dépendances

objectif : regrouper les bibliothèques Python nécessaires à l’analyse exploratoire des données.

In [1]:
import pandas as pd
import numpy as np


## 2. Chargement du fichier Excel
Objectif : charger le fichier Excel et vérifier l'accès aux différentes feuilles.

In [2]:
# chargement du fichier
file_path = "../data/raw/CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xls"

xls = pd.ExcelFile(file_path)


## 3. Chargement de la feuille 1 — CONTACTS DIRECTEUR-DRH-SERVICE
Objectif : charger la première feuille et vérifier sa structure (dimensions, colonnes).


In [3]:
# afficher les noms des feuilles
xls.sheet_names


['CONTACTS DIRECTEUR-DRH-SERVICE ', 'Taxe', 'Pour e-mailing']

In [4]:
# charger la feuille "CONTACTS DIRECTEUR-DRH-SERVICE "  dans un DataFrame
df_contacts = pd.read_excel(
    "../data/raw/CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xls",
    sheet_name="CONTACTS DIRECTEUR-DRH-SERVICE "
)


In [5]:
# afficher la taille du DataFrame
df_contacts.shape


(6799, 23)

In [6]:
# afficher les colonnes du DataFrame
df_contacts.columns


Index(['Code.Entité', 'Libellé.Entité', 'Code.Type d'entité',
       'Assujetti.Entité', 'Rue (ligne 1).Adresse', 'Rue (ligne 2).Adresse',
       'Rue (ligne 3).Adresse', 'Rue (ligne 4).Adresse', 'Code postal.Ville',
       'Nom.Ville', 'Code.Type d'adresse',
       'Nombre de stage (sans contrat pro)', 'Noms des stagiaires',
       'Nombre de contrat pro', 'Noms des alternants contrats pro',
       'Nombre apprentissage', 'Noms des apprentis', 'Code.Type d'événement',
       'Montant global.Taxe versement', 'Libellé.Titre', 'Nom.Individu',
       'Prénom.Individu', 'Coordonnée.Coordonnée'],
      dtype='str')

Observations :
- La feuille contient plus d’informations que prévu initialement, avec des dimensions “formation / taxe / événements” mélangées aux contacts :
    - je trouve bien les colonnes entité / adresse / contact → visibles initialement
    - je trouve aussi les colonnes stage / alternance / taxe / événement → non visibles initialement

In [7]:
# afficher les types de données des colonnes
df_contacts.dtypes


Code.Entité                               str
Libellé.Entité                            str
Code.Type d'entité                        str
Assujetti.Entité                      float64
Rue (ligne 1).Adresse                     str
Rue (ligne 2).Adresse                  object
Rue (ligne 3).Adresse                     str
Rue (ligne 4).Adresse                     str
Code postal.Ville                      object
Nom.Ville                                 str
Code.Type d'adresse                       str
Nombre de stage (sans contrat pro)    float64
Noms des stagiaires                       str
Nombre de contrat pro                 float64
Noms des alternants contrats pro          str
Nombre apprentissage                  float64
Noms des apprentis                        str
Code.Type d'événement                     str
Montant global.Taxe versement         float64
Libellé.Titre                             str
Nom.Individu                           object
Prénom.Individu                   

Observations :
  - les colonnes de comptage ('Nombre de stage', Nombre de contrat pro', etc) sont en float 64 -> probable présence de NaN
  - 'Assujetti.Entité est en float 64 -> booléen 'déguisé' (1/NaN au lieu de 1/0) -> quand c'est false, c'est remplacé par NaN
  - 'Code postal.Ville' en object -> probablement à cause des codes postaux avec zéros, CEDEX...
  - cette feuille mélange plusieurs dimensions métier dans une même table : des données de contact / des données pédagogiques liées à des dispositifs (stage, contrats pro, alternance, etc.) / des données administratives (taxe).

## 4. Valeurs manquantes

Objectif : mesurer la qualité des données par colonne afin d’identifier les champs critiques.


In [8]:
# afficher le nombre de valeurs manquantes par colonne
missing_count = df_contacts.isna().sum()
missing_count


Code.Entité                              0
Libellé.Entité                           0
Code.Type d'entité                      49
Assujetti.Entité                       331
Rue (ligne 1).Adresse                    2
Rue (ligne 2).Adresse                 4253
Rue (ligne 3).Adresse                 6488
Rue (ligne 4).Adresse                 6790
Code postal.Ville                        0
Nom.Ville                                0
Code.Type d'adresse                      0
Nombre de stage (sans contrat pro)    4368
Noms des stagiaires                   4368
Nombre de contrat pro                 4465
Noms des alternants contrats pro      4465
Nombre apprentissage                  5161
Noms des apprentis                    5161
Code.Type d'événement                 5785
Montant global.Taxe versement         5785
Libellé.Titre                           10
Nom.Individu                             0
Prénom.Individu                          1
Coordonnée.Coordonnée                  307
dtype: int6

In [9]:
# afficher le pourcentage de valeurs manquantes par colonne
missing_percent = (df_contacts.isna().mean() * 100).round(2)
missing_percent


Code.Entité                            0.00
Libellé.Entité                         0.00
Code.Type d'entité                     0.72
Assujetti.Entité                       4.87
Rue (ligne 1).Adresse                  0.03
Rue (ligne 2).Adresse                 62.55
Rue (ligne 3).Adresse                 95.43
Rue (ligne 4).Adresse                 99.87
Code postal.Ville                      0.00
Nom.Ville                              0.00
Code.Type d'adresse                    0.00
Nombre de stage (sans contrat pro)    64.24
Noms des stagiaires                   64.24
Nombre de contrat pro                 65.67
Noms des alternants contrats pro      65.67
Nombre apprentissage                  75.91
Noms des apprentis                    75.91
Code.Type d'événement                 85.09
Montant global.Taxe versement         85.09
Libellé.Titre                          0.15
Nom.Individu                           0.00
Prénom.Individu                        0.01
Coordonnée.Coordonnée           

In [10]:
# créer un DataFrame récapitulatif des valeurs manquantes
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_summary


,missing_count,missing_percent
Rue (ligne 4).Adresse,6790,99.87
Rue (ligne 3).Adresse,6488,95.43
Montant global.Taxe versement,5785,85.09
Code.Type d'événement,5785,85.09
Noms des apprentis,5161,75.91
Nombre apprentissage,5161,75.91
Noms des alternants contrats pro,4465,65.67
Nombre de contrat pro,4465,65.67
Noms des stagiaires,4368,64.24
Nombre de stage (sans contrat pro),4368,64.24


### Lecture des valeurs manquantes

L’analyse met en évidence plusieurs niveaux de complétude :

- Certaines colonnes d’adresse (lignes 3 et 4) sont quasiment vides (> 95 % de valeurs manquantes), ce qui limite fortement leur exploitabilité.
- Les colonnes liées à la formation (stages, contrats pro, apprentissages) et à la taxe d'apprentissage présentent des taux de valeurs manquantes élevés, cohérents avec le fait que ces informations ne concernent qu’une partie des entités.
- Les colonnes essentielles à l'identification des entités et des individus (identité, localisation, nom/prénom) sont très majoritairement complètes.
- La colonne `Coordonnée.Coordonnée` joue un rôle central pour les usages CRM. Elle regroupe plusieurs types d’informations : adresses e-mail, numéros de téléphone, ainsi que des valeurs textuelles indiquant le statut de contact (absence d’e-mail, désinscription, refus d’e-mailing, NC, etc.). Cette hétérogénéité met en évidence la nécessité, lors de la phase ETL, de distinguer les coordonnées par type (e-mail, téléphone) et de conserver explicitement l’information liée à l’autorisation ou non de l’e-mailing afin de permettre des usages CRM adaptés.

## 5. Analyse du contenu de la colonne Coordonnée.Coordonnée

Objectif : quantifier les différents types d’informations présents
(adresses e-mail, numéros de téléphone, statuts).


In [11]:
# normalisation minimale de la colonne Coordonnée.Coordonnée
# (conversion en texte et suppression des espaces) afin de permettre l’analyse
coord = df_contacts["Coordonnée.Coordonnée"].astype(str).str.strip()


In [12]:
# détection approximative des emails (présence du caractère '@')
mask_email = coord.str.contains(r"@", na=False)


In [13]:
# détection approximative des numéros de téléphone
# (présence d’une suite de chiffres)
mask_phone = coord.str.contains(r"\d{2}.*\d{2}", na=False)


In [14]:
# détection des valeurs ne correspondant ni à un email ni à un téléphone
mask_status = ~(mask_email | mask_phone)


In [15]:
# comptage du nombre de lignes correspondant à chaque type de coordonnée
email_count = mask_email.sum()
phone_count = mask_phone.sum()
status_count = mask_status.sum()

email_count, phone_count, status_count


(np.int64(6477), np.int64(21), np.int64(314))

In [16]:
# création d’un tableau de synthèse présentant
# le volume et la proportion de chaque type de coordonnée
coord_summary = pd.DataFrame({
    "type": ["email", "téléphone", "statut / autre"],
    "count": [email_count, phone_count, status_count],
    "percent": [
        round(email_count / len(df_contacts) * 100, 2),
        round(phone_count / len(df_contacts) * 100, 2),
        round(status_count / len(df_contacts) * 100, 2),
    ]
})

coord_summary


,type,count,percent
0,email,6477,95.26
1,téléphone,21,0.31
2,statut / autre,314,4.62


Texte des statuts observés dans la colonne `Coordonnée.Coordonnée` :

    PAS DE CAMPAGNE E-MAILING

    PAS POUR CAMPAGNE E-MAILING

    PAS D'ENVOI E-MAILING

    PAS D'EMAILING

    DESINSCRIT E-MAILING

    DESINSCRIPTION E-MAILING

    DÉSINSCRIPTION E-MAILING

    DESABONNEMENT E-MAILING

    NC

Ces valeurs textuelles indiquent des refus explicites d’e-mailing ou des informations de non-communication. Elles sont hétérogènes dans leur formulation mais portent des significations proches.

### Analyse du contenu de la colonne Coordonnée.Coordonnée

La colonne `Coordonnée.Coordonnée` jour un rôle central pour les usages CRM. Elle regroupe plusieurs tupes d'informations : environ 95% des lignes sont adresses e-mail, une très faible proportion correspond à des numéros de téléphone, tandis qu’une part non négligeable des valeurs correspond à des informations de statut (absence d’e-mail, refus ou désinscription à l’e-mailing, NC, etc.).

Ces résultats confirment l’intérêt de :
- distinguer les coordonnées par type (e-mail, téléphone),
- conserver explicitement l’information relative à l’autorisation ou non de l’e-mailing, afin de permettre des usages CRM conformes et adaptés.


## 6. Analyse des doublons email

Objectif : identifier la présence éventuelle de doublons d'adresses email, afin d'évaluer la qualité de la base pour un usage CRM/emailing.

In [17]:
# préparation de la colonne email :
# extraction de la colonne Coordonnée.Coordonnée
# et normalisation minimale pour l’analyse des doublons
emails = (
    df_contacts["Coordonnée.Coordonnée"]
    .astype(str)          # conversion en texte pour éviter les erreurs liées aux NaN
    .str.strip()          # suppression des espaces en début / fin
    .str.lower()          # mise en minuscules pour éviter les faux doublons
)


In [18]:
# sélection des lignes contenant une adresse e-mail (pour les isoler)
emails_only = emails[emails.str.contains("@", na=False)]


In [19]:
# détection des emails apparaissant plus d'une fois (doublons)
email_duplicates = emails_only[emails_only.duplicated(keep=False)]


In [20]:
# Quantification des doublons email :

# nombre total d'emails
total_emails = emails_only.shape[0]

# nombre d'emails distincts
unique_emails = emails_only.nunique()

# nombre d'emails dupliqués (en volume)
duplicate_emails_count = email_duplicates.shape[0]

total_emails, unique_emails, duplicate_emails_count


(6477, 5788, 1139)

In [21]:
# calcul du taux de doublons email
duplicate_rate = round((duplicate_emails_count / total_emails) * 100, 2)
duplicate_rate


17.59

In [22]:
# comptage des emails les plus fréquemment présents
email_duplicate_summary = email_duplicates.value_counts().head(10)
email_duplicate_summary


Coordonnée.Coordonnée
thales.alternance-stage@pontoonsolutions.com    18
admin-stg-alt.navalgroup@manpowergroup.fr       13
stage-alternance@arkea.com                      13
stagesalternances.obssa@orange.com              12
admin-stg-alt.navalgroup@tapfin.fr              12
valerie.sable@capgemini.com                     11
eidprhecoles@e-i.com                            10
welcome.earlycareers@airbus.com                  9
sandra.belliure@capgemini.com                    8
formation.bretagne@ifremer.fr                    8
Name: count, dtype: int64

### Analyse des doublons d’adresses e-mail

L’analyse des adresses e-mail montre la présence de doublons dans la base.
Ces doublons peuvent correspondre :
- à des adresses génériques partagées (ex. contact@, info@),
- à des contacts communs à plusieurs entités ou services.

Ce point devra être pris en compte lors de la phase ETL, afin de définir les règles de gestion des contacts (unicité, rattachement aux entités, priorisation des coordonnées).

L’analyse des doublons montre que ceux-ci sont majoritairement associés aux contextes de stage et d’alternance. Cela suggère l’utilisation d’adresses e-mail communes ou génériques pour le suivi administratif de plusieurs stagiaires ou alternants.

Ces doublons ne traduisent donc pas nécessairement une mauvaise qualité des données, mais reflètent des pratiques métier spécifiques. Ce point devra être pris en compte lors de la définition des règles d’unicité et de rattachement des contacts dans le CRM.



## 7. Qualité des adresses e-mail

Objectif : évaluer la qualité des adresses e-mail (format, valeurs aberrantes) afin d’anticiper les règles de validation et de nettoyage lors de la phase ETL.

- Extraction + normalisation minimale

In [23]:
# extraction de la colonne coordonnée et filtrage des lignes qui ressemblent à des emails
emails_raw = df_contacts["Coordonnée.Coordonnée"].astype(str)

# normalisation minimale pour l'analyse (sans transformation définitive)
emails = emails_raw.str.strip().str.lower()

# on ne garde que les valeurs contenant '@' (heuristique EDA)
emails = emails[emails.str.contains("@", na=False)]


- Contrôles simples de qualité

In [24]:
# emails vides après strip (rare mais possible)
empty_after_strip = (emails.str.len() == 0).sum()

# emails contenant des espaces internes (souvent signe de mauvaise saisie)
with_spaces = emails.str.contains(r"\s", regex=True).sum()

# emails contenant plusieurs '@' (format suspect)
multiple_at = (emails.str.count("@") > 1).sum()

# emails sans point dans la partie domaine (= heuristique simple)
no_dot_after_at = (~emails.str.split("@").str[1].str.contains(r"\.", na=False)).sum()

empty_after_strip, with_spaces, multiple_at, no_dot_after_at


(np.int64(0), np.int64(207), np.int64(4), np.int64(5))

- Contrôle complémentaire : repérer les valeurs où un statut est concaténé à l’adresse e-mail sans séparateur (ex. `.comPAS...`).

In [25]:
# contrôle complémentaire : statut concaténé juste après un domaine email (ex: .comPAS...)
mask_concat_status = emails.str.contains(r"\.(com|fr|net|org|eu)[A-Za-z]", regex=True)

concat_status_count = mask_concat_status.sum()
concat_status_percent = round(concat_status_count / len(emails) * 100, 2)

concat_status_count, concat_status_percent

/tmp/ipykernel_104328/1875381148.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_concat_status = emails.str.contains(r"\.(com|fr|net|org|eu)[A-Za-z]", regex=True)


(np.int64(80), np.float64(1.24))

- Taux

In [26]:
# total d'emails analysés
n = len(emails)

# (re)calcul du % pour le contrôle "statut concaténé" sur la même base n
concat_status_percent = round(concat_status_count / n * 100, 2)

quality_summary = pd.DataFrame({
    "check": [
        "vides après strip",
        "contient des espaces",
        "plusieurs @",
        "pas de '.' après @ (heur.)",
        "statut concaténé après domaine (ex: .comPAS...)",
    ],
    "count": [
        empty_after_strip,
        with_spaces,
        multiple_at,
        no_dot_after_at,
        concat_status_count,
    ],
    "percent": [
        round(empty_after_strip/n*100, 2),
        round(with_spaces/n*100, 2),
        round(multiple_at/n*100, 2),
        round(no_dot_after_at/n*100, 2),
        concat_status_percent,
    ]
})

quality_summary


,check,count,percent
0,vides après strip,0,0.00
1,contient des espaces,207,3.20
2,plusieurs @,4,0.06
3,pas de '.' après @ (heur.),5,0.08
4,statut concaténé après domaine (ex: .comPAS...),80,1.24


- Exemples concrets

In [27]:
# exemples d'emails avec espaces
emails[emails.str.contains(r"\s", regex=True)].head(10)


40     olivier.adraste@atempo.com-pas de campagne e-m...
102    marie-clotildealvaro@gmail.compas de campagne ...
185    anthony.artus@engie.com - pas de campagne e-ma...
213    loraine.audeguy@dsia.com-pas de campagne e-mai...
227    stephanie.auffray@logica.com - pas de campagne...
271          cbabonneau@asi.fr-pas de campagne e-mailing
327    ol.barbreau@bouygues-es.com-pas d'envoi e-mailing
340    lea.barratt@orange.com - pas de campagne e-mai...
378    marie-christine.baudrier@st.com-pas de campagn...
407    morgane.bec.external@airbus.com - pas de campa...
Name: Coordonnée.Coordonnée, dtype: str

In [28]:
# exemples d'emails sans '.' dans le domaine
emails[~emails.str.split("@").str[1].str.contains(r"\.", na=False)].head(10)


858                   matthieu@evernet
1195            nicolas.cariou@gwagenn
1543      mathieu@collignon@airbus.com
3207    mhuchet@@mobilitytechgreen.com
5278      alpenfor@bouyguestelecom@.fr
Name: Coordonnée.Coordonnée, dtype: str

In [29]:
# exemples de valeurs détectées (email + statut concaténé)
emails[mask_concat_status].head(10)


22                             hr.france@ctingenierie.com
102     marie-clotildealvaro@gmail.compas de campagne ...
591     jessica.bezier@thalesgroup.compas de campagne ...
595     jessica.bezier@thalesgroup.compas de campagne ...
596     jessica.bezier@thalesgroup.compas de campagne ...
846            b.boulo@kiabi.compas de campagne e-mailing
954     f.brault@laroutedescomptoirs.compas de campagn...
1000                          azur.nettoyage134@orange.fr
1026     pauline.brun@danone.compas de campagne e-mailing
1558                           yannick.combaud@airbus.com
Name: Coordonnée.Coordonnée, dtype: str

### Analyse des adresses e-mail

La majorité des adresses e-mail ont une structure exploitable, mais plusieurs anomalies impactent l’usage CRM / e-mailing.

- Le cas le plus fréquent concerne la présence d’espaces dans la valeur (≈ 3 %), signe de saisie non normalisée.
- On observe aussi des cas où un statut est concaténé directement à l’adresse e-mail (ex. `.comPAS...`), ce qui rend l’adresse inutilisable telle quelle pour une campagne.
- Les anomalies de format plus “techniques” (plusieurs `@`, domaine incomplet) restent minoritaires.

Ces constats confirment que la colonne `Coordonnée.Coordonnée` contient à la fois des coordonnées et des informations de statut, et qu’il faudra les distinguer pour produire une donnée de contact exploitable.


## 8. Préparation de la structuration des données pour intégration

Objectif : À l’issue de l’EDA, certaines règles de structuration peuvent être définies afin de préparer l’intégration des données dans MongoDB et leur affichage dans le CRM.
Cette section formalise la structure cible des données à partir des observations réalisées, sans transformation définitive à ce stade.


Champs identifiés à partir de l’EDA : 

- Entité
    -  Code entité
    - Libellé entité
    - Type d’entité
    - Assujetti à la taxe

- Adresse
    - Ligne principale
    - Complément d’adresse (issues des lignes 2, 3, 4)
    - Code postal
    - Ville

- Contact
    - Civilité
    - Nom
    - Prénom

- Coordonnées
    - Email (lorsqu’identifié)
    - Téléphone (lorsqu’identifié)
    - Valeur brute originale (Coordonnée.Coordonnée)

- Consentement
    - Newsletter
    - Emailing
    - Publicité
    - Ancien contact
    - Raison du refus
    - Texte source du statut

Remarque : Les champs de consentement sont déduits des valeurs textuelles observées dans la colonne `Coordonnée.Coordonnée`.

## 9. Table de correspondance des statuts de contact
Objectif : Cette table permet de relier les formulations textuelles observées dans les données sources à des champs structurés de consentement.

Table de correspondance :

| Texte observé              | newsletter | emailing | publicite | ancien_contact | raison         |
| -------------------------- | ---------- | -------- | --------- | -------------- | -------------- |
| PAS DE CAMPAGNE E-MAILING  | null       | false    | false     | false          | refus_emailing |
| PAS D'ENVOI E-MAILING      | null       | false    | false     | false          | refus_emailing |
| DESINSCRIPTION E-MAILING   | false      | false    | false     | false          | desinscription |
| DÉSINSCRIPTION E-MAILING   | false      | false    | false     | false          | desinscription |
| DESABONNEMENT E-MAILING    | false      | false    | false     | false          | desinscription |
| NC                         | null       | null     | null      | null           | non_communique |
| *(à venir)* ANCIEN CONTACT | false      | false    | false     | true           | ancien_contact |

Remarque : La catégorie *ancien contact* n’est pas présente dans ce fichier mais est anticipée pour les jeux de données suivants.

In [30]:
# table de correspondance : texte source -> règles de consentement
STATUS_MAP = {
    "PAS DE CAMPAGNE E-MAILING": {
        "newsletter": None,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "refus_emailing",
    },
    "PAS POUR CAMPAGNE E-MAILING": {
        "newsletter": None,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "refus_emailing",
    },
    "PAS D'ENVOI E-MAILING": {
        "newsletter": None,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "refus_emailing",
    },
    "PAS D'EMAILING": {
        "newsletter": None,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "refus_emailing",
    },
    "DESINSCRIT E-MAILING": {
        "newsletter": False,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "desinscription",
    },
    "DESINSCRIPTION E-MAILING": {
        "newsletter": False,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "desinscription",
    },
    "DÉSINSCRIPTION E-MAILING": {
        "newsletter": False,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "desinscription",
    },
    "DESABONNEMENT E-MAILING": {
        "newsletter": False,
        "emailing": False,
        "publicite": False,
        "ancien_contact": False,
        "raison": "desinscription",
    },
    "NC": {
        "newsletter": None,
        "emailing": None,
        "publicite": None,
        "ancien_contact": None,
        "raison": "non_communique",
    },
}

STATUS_MAP.keys()


dict_keys(['PAS DE CAMPAGNE E-MAILING', 'PAS POUR CAMPAGNE E-MAILING', "PAS D'ENVOI E-MAILING", "PAS D'EMAILING", 'DESINSCRIT E-MAILING', 'DESINSCRIPTION E-MAILING', 'DÉSINSCRIPTION E-MAILING', 'DESABONNEMENT E-MAILING', 'NC'])

## 10. Application de la table de correspondance aux coordonnées

Objectif : appliquer la table de correspondance à la colonne `Coordonnée.Coordonnée` afin d’extraire des champs structurés (email, téléphone, statut) et vérifier que les règles de correspondance fonctionnent.


In [31]:
import re

# Liste des statuts reconnus à partir de la table de correspondance
KNOWN_STATUSES = list(STATUS_MAP.keys())


# Fonction de normalisation textuelle
def normalize_text(value: str) -> str:
    """
    Normalise une chaîne de caractères afin de faciliter les comparaisons :
    - suppression des espaces en début/fin
    - passage en majuscules
    - normalisation simple des caractères accentués
    """
    if value is None:
        return ""

    text = str(value).strip().upper()
    text = (
        text.replace("É", "E")
            .replace("È", "E")
            .replace("Ê", "E")
            .replace("À", "A")
            .replace("Â", "A")
            .replace("Î", "I")
            .replace("Ï", "I")
            .replace("Ô", "O")
            .replace("Û", "U")
            .replace("Ü", "U")
            .replace("Ç", "C")
    )
    return text


# Association entre texte normalisé et libellé d'origine du statut
STATUS_KEY_BY_NORMALIZED = {
    normalize_text(status): status for status in KNOWN_STATUSES
}

# Expressions régulières pour la détection des coordonnées
EMAIL_PATTERN = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
PHONE_PATTERN = re.compile(r"(\+33|0)\s*[1-9](?:[\s\.\-]?\d{2}){4}")


# Fonction d'analyse détaillée d'une valeur de coordonnée
def parse_coordonnee(value):
    """
    Analyse une valeur issue de la colonne 'Coordonnée.Coordonnée' et renvoie :
    - la valeur brute originale
    - l'adresse e-mail détectée (le cas échéant)
    - le numéro de téléphone détecté (le cas échéant)
    - le statut textuel reconnu
    - les informations de consentement associées
    """
    raw_value = "" if value is None else str(value).strip()

    if raw_value == "":
        return {
            "raw": raw_value,
            "email": None,
            "telephone": None,
            "statut_source": None,
            "consent": None,
        }

    # Extraction de l'adresse e-mail
    email_match = EMAIL_PATTERN.search(raw_value)
    email = email_match.group(0).lower() if email_match else None

    # Extraction du numéro de téléphone
    phone_match = PHONE_PATTERN.search(raw_value)
    telephone = phone_match.group(0) if phone_match else None

    # Détection d'un statut textuel
    normalized_raw = normalize_text(raw_value)
    statut_source = None

    if normalized_raw in STATUS_KEY_BY_NORMALIZED:
        statut_source = STATUS_KEY_BY_NORMALIZED[normalized_raw]
    else:
        for normalized_status, original_status in STATUS_KEY_BY_NORMALIZED.items():
            if normalized_status in normalized_raw:
                statut_source = original_status
                break

    # Application de la table de correspondance si un statut est reconnu
    consent = None
    if statut_source in STATUS_MAP:
        consent = STATUS_MAP[statut_source].copy()
        consent["source_text"] = statut_source

    return {
        "raw": raw_value,
        "email": email,
        "telephone": telephone,
        "statut_source": statut_source,
        "consent": consent,
    }


# Application de la fonction sur un échantillon pour validation
sample = df_contacts["Coordonnée.Coordonnée"].dropna().sample(10, random_state=42)
parsed_sample = sample.apply(parse_coordonnee)

pd.DataFrame(list(parsed_sample)) # conversion en DataFrame pour affichage


,raw,email,telephone,statut_source,consent
0,c.villedieu@rainette-ecologie.com,c.villedieu@rainette-ecologie.com,None,NaN,None
1,hugo.fernandez@enedis.fr,hugo.fernandez@enedis.fr,None,NaN,None
2,administratif@hera-mi.com,administratif@hera-mi.com,None,NaN,None
3,xavier.marjou@orange.com,xavier.marjou@orange.com,None,NaN,None
4,pascal.coince@sercel.com,pascal.coince@sercel.com,None,NC,"{'newsletter': None, 'emailing': None, 'public..."
5,allan.brustolin@reseau.sncf.fr,allan.brustolin@reseau.sncf.fr,None,NC,"{'newsletter': None, 'emailing': None, 'public..."
6,ajezequel@asi.fr,ajezequel@asi.fr,None,NaN,None
7,jean-francois.lemoine@soprasteria.com,jean-francois.lemoine@soprasteria.com,None,NC,"{'newsletter': None, 'emailing': None, 'public..."
8,dimitri.ho@wavestone.com,dimitri.ho@wavestone.com,None,NaN,None
9,ulysse.cadour@infotel.com,ulysse.cadour@infotel.com,None,NaN,None


## 11. Génération d’un JSON intermédiaire des contacts

Objectif : générer une structure JSON intermédiaire à partir des données analysées, en vue de leur intégration dans MongoDB et de leur affichage dans le CRM.


In [32]:
import json

# --- Transformation des données en documents JSONL ---

documents = []

# Construction des documents contacts
for _, row in df_contacts.iterrows():
    coord = parse_coordonnee(row["Coordonnée.Coordonnée"])

    # Construction du document JSON
    doc = {
        "entite": {
            "code": row.get("Code.Entité"),
            "libelle": row.get("Libellé.Entité"),
            "type": row.get("Code.Type d'entité"),
            "assujetti_taxe": row.get("Assujetti.Entité"),
        },
        "adresse": {
            "ligne1": row.get("Rue (ligne 1).Adresse"),
            "complement": " ".join(
                str(v) for v in [
                    row.get("Rue (ligne 2).Adresse"),
                    row.get("Rue (ligne 3).Adresse"),
                    row.get("Rue (ligne 4).Adresse"),
                ]
                if v and str(v).strip()
            ) or None,
            "code_postal": row.get("Code postal.Ville"),
            "ville": row.get("Nom.Ville"),
        },
        "contact": {
            "civilite": row.get("Libellé.Titre"),
            "nom": row.get("Nom.Individu"),
            "prenom": row.get("Prénom.Individu"),
        },
        "coordonnees": {
            "email": coord["email"],
            "telephone": coord["telephone"],
            "raw": coord["raw"],
        },
        "consent": coord["consent"],
    }

    documents.append(doc)

# --- Export en JSONL ---

output_path = "contacts_intermediaire.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

output_path, len(documents)


('contacts_intermediaire.jsonl', 6799)

In [33]:
# Affichage des 3 premières lignes du fichier JSONL généré pour vérification
with open("contacts_intermediaire.jsonl", encoding="utf-8") as f:
    for _ in range(3):
        print(next(f))


{"entite": {"code": "CYRISEA", "libelle": "CYRISEA", "type": "ENTREPRISE", "assujetti_taxe": NaN}, "adresse": {"ligne1": "15 Av. du Professeur Jean Rouxel", "complement": "nan nan nan", "code_postal": 44481, "ville": "CARQUEFOU"}, "contact": {"civilite": "Monsieur", "nom": " BERUTTO", "prenom": "Richard"}, "coordonnees": {"email": "richard.berutto@cyrisea.com", "telephone": null, "raw": "richard.berutto@cyrisea.com"}, "consent": null}

{"entite": {"code": "LA_MAISON_D_AUTREFOIS", "libelle": "LA MAISON D'AUTREFOIS_29", "type": "ENTREPRISE", "assujetti_taxe": 1.0}, "adresse": {"ligne1": "7 rue de l'église", "complement": "nan nan nan", "code_postal": 29290, "ville": "SAINT RENAN"}, "contact": {"civilite": "Madame", "nom": " EL HACHIMI EL IDRISSI ", "prenom": "Emmanuelle "}, "coordonnees": {"email": null, "telephone": null, "raw": "nan"}, "consent": null}

{"entite": {"code": "THALES_SYSTEMES_AEROPORTES_29", "libelle": "THALES DMS FRANCE ", "type": "ENTREPRISE", "assujetti_taxe": 1.0}, "a

## 12. Conclusion – Feuille 1 : CONTACTS DIRECTEUR-DRH-SERVICE


Cette première feuille fournit une base riche d’informations sur les entités et les contacts associés
(identité des structures, localisation, interlocuteurs, coordonnées).

L’analyse met cependant en évidence une forte hétérogénéité des informations regroupées dans une même ligne :
données de contact, informations liées à la formation (stages, alternance, apprentissage) et éléments liés à
l’e-mailing ou à la taxe d’apprentissage coexistent sans séparation claire.

En particulier, la colonne `Coordonnée.Coordonnée` ne correspond pas à une donnée unique : elle regroupe des
adresses e-mail, des numéros de téléphone et des informations de statut (refus d’e-mailing, désinscription),
parfois concaténées à l’adresse elle-même. Ces pratiques rendent les données difficiles à exploiter directement
dans un contexte CRM.

Cette feuille constitue donc une source précieuse, mais nécessite une structuration préalable afin de dissocier
les différents types d’informations et de rendre les contacts réellement exploitables pour des usages CRM et
marketing.

Les éléments de structuration et de transformation présentés ci-dessus s’appuient directement sur les constats issus de cette analyse exploratoire.


# EDA – Feuille 2 : Taxe

### Objectif
Analyser la feuille « Taxe » afin de :
- vérifier la structure et la qualité des données (emails, doublons, valeurs manquantes),
- mesurer le recouvrement avec la feuille 1 (contacts),
- déterminer comment cette feuille peut enrichir les données existantes (information “taxe 2025”),
- générer un JSONL intermédiaire.


## 1. Chargement et structure
Objectif : charger la feuille 2 "Taxe" et vérifier sa structure (nombre de lignes, de colonnes, nom des colonnes).

In [34]:
# Chargement de la feuille 2
df_taxe = pd.read_excel(file_path, sheet_name="Taxe")

# Vérification de la structure : dimensions et nom de colonne
df_taxe.shape, df_taxe.columns


((101, 1), Index(['Verseurs de taxe 2025'], dtype='str'))

## 2. Contenu : aperçu, types et valeurs manquantes

Objectif : vérifier si ce sont uniquement des emails, s'il y a des valeurs manquantes et des doublons.

In [35]:
# Affichage des premières lignes pour valider le contenu
# (emails attendus)
df_taxe.head(10)


,Verseurs de taxe 2025
0,info@cristec.fr
1,a.combier@arcds.fr
2,admin@dmgmori-sailingteam.com
3,alexandra.lafrogne@naval-group.com
4,amani.jebali@capgemini.com
5,amaury.le-hir-reynaud@arkea.com
6,anais.bernard@entech-se.com
7,anne-claire.magueur@suravenir.fr
8,"antoine.millot@ett-hvac.com, contact@ett-hvac.com"
9,antoine.vignonvif.fr


In [36]:
# Type des données (utile pour repérer une anomalie)
df_taxe.dtypes


Verseurs de taxe 2025    str
dtype: object

In [37]:
# Valeurs manquantes (lignes vides ou cellules vides)
missing_count = df_taxe.isna().sum()
missing_percent = (df_taxe.isna().mean() * 100).round(2)

pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})


,missing_count,missing_percent
Verseurs de taxe 2025,0,0.0


## 3. Normalisation minimale

Objectif : supprimer les valeurs manquantes, convertir en chaine de caractères, supprimer les espaces en début et fin de chaîne, passer en minuscules pour obtenir une comparaison fiable des adresses e-mail.

In [38]:
# normalisation minimale des emails
emails_taxe = (
    df_taxe['Verseurs de taxe 2025']
    .dropna()      # suppression des valeurs manquantes
    .astype(str)   # conversion en texte (éviter les erreurs liées aux NaN)
    .str.strip()   # suppression des espaces en début / fin
    .str.lower()   # mise en minuscules (pour comparaison fiable)
)

emails_taxe.head()


0                       info@cristec.fr
1                    a.combier@arcds.fr
2         admin@dmgmori-sailingteam.com
3    alexandra.lafrogne@naval-group.com
4            amani.jebali@capgemini.com
Name: Verseurs de taxe 2025, dtype: str

## 4. Analyse des doublons d’adresses e-mail
Objectif : identifier la présence éventuelle de doublons d'adresses email.

In [39]:
# analyse des doublons
total_emails = len(emails_taxe)
unique_emails = emails_taxe.nunique()
duplicate_count = total_emails - unique_emails
duplicate_rate = round((duplicate_count / total_emails) * 100, 2)

pd.DataFrame([{
    "emails_total": total_emails,
    "emails_uniques": unique_emails,
    "doublons": duplicate_count,
    "taux_doublons_%": duplicate_rate
}])


,emails_total,emails_uniques,doublons,taux_doublons_%
0,101,98,3,2.97


In [40]:
# Liste des emails dupliqués (pour contrôle)
emails_taxe[emails_taxe.duplicated(keep=False)].sort_values()


19    clientargel@argel.fr
20    clientargel@argel.fr
36    eidprhecoles@e-i.com
37    eidprhecoles@e-i.com
0          info@cristec.fr
50         info@cristec.fr
Name: Verseurs de taxe 2025, dtype: str

## 5. Contrôles de qualité des adresses e-mail

Objectif : évaluer la qualité des adresses e-mail (format, valeurs aberrantes) afin d’anticiper les règles de validation et de nettoyage lors de la phase ETL.

In [41]:
# Contrôle 1 : emails vides après normalisation
# (simple vérification, car avec dropna, en principe 0)
empty_after_strip = (emails_taxe.str.len() == 0).sum()

# Contrôle 2 : présence d'espaces à l'intérieur de l'adresse (anomalie)
contains_inner_spaces = emails_taxe.str.contains(r"\s", na=False).sum()

# Contrôle 3 : plusieurs @ (anomalie)
multiple_at = (emails_taxe.str.count("@") > 1).sum()

# Contrôle 4 : absence de '.' après @ (heuristique simple)
no_dot_after_at = emails_taxe[~emails_taxe.str.split("@").str[1].str.contains(r"\.", na=False)].shape[0]

pd.DataFrame([
    {"check": "vides après strip", "count": int(empty_after_strip), "percent": round(empty_after_strip/total_emails*100, 2)},
    {"check": "contient des espaces", "count": int(contains_inner_spaces), "percent": round(contains_inner_spaces/total_emails*100, 2)},
    {"check": "plusieurs @", "count": int(multiple_at), "percent": round(multiple_at/total_emails*100, 2)},
    {"check": "pas de '.' après @", "count": int(no_dot_after_at), "percent": round(no_dot_after_at/total_emails*100, 2)},
])


,check,count,percent
0,vides après strip,0,0.00
1,contient des espaces,2,1.98
2,plusieurs @,2,1.98
3,pas de '.' après @,1,0.99


In [42]:
# Exemples des anomalies détectées (si existantes)
examples_with_spaces = emails_taxe[emails_taxe.str.contains(r"\s", na=False)].head(10)
examples_multiple_at = emails_taxe[emails_taxe.str.count("@") > 1].head(10)
examples_no_dot = emails_taxe[~emails_taxe.str.split("@").str[1].str.contains(r"\.", na=False)].head(10)

examples_with_spaces, examples_multiple_at, examples_no_dot


(8     antoine.millot@ett-hvac.com, contact@ett-hvac.com
 13    catherine.gourrierec@atos.net, campusfrance@at...
 Name: Verseurs de taxe 2025, dtype: str,
 8     antoine.millot@ett-hvac.com, contact@ett-hvac.com
 13    catherine.gourrierec@atos.net, campusfrance@at...
 Name: Verseurs de taxe 2025, dtype: str,
 9    antoine.vignonvif.fr
 Name: Verseurs de taxe 2025, dtype: str)

## 6. Recouvrement avec la feuille 1 (contacts)

Objectif : mesurer le recouvrement des adresses e-mail entre la feuille 2 (taxe) et la feuille 1 (contacts) afin d’évaluer comment cette feuille peut enrichir les données existantes.

In [43]:
# Extraction des emails présents dans la feuille 1 
# (à partir de la colonne 'Coordonnée.Coordonnée')
emails_f1 = (
    df_contacts["Coordonnée.Coordonnée"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.extract(r'([a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,})')[0]
    .dropna()
    .unique()
)

nb_emails_f1 = len(emails_f1)

# Recouvrement feuille 2 / feuille 1
emails_communs = set(emails_taxe) & set(emails_f1)
nb_communs = len(emails_communs)

# Emails présents dans Taxe mais absents de la feuille 1
emails_nouveaux = set(emails_taxe) - set(emails_f1)
nb_nouveaux = len(emails_nouveaux)

# Résumé du recouvrement
pd.DataFrame([{
    "emails_feuille_1_uniques": nb_emails_f1,
    "emails_taxe_uniques": unique_emails,
    "emails_communs": nb_communs,
    "emails_nouveaux_taxe": nb_nouveaux
}])


,emails_feuille_1_uniques,emails_taxe_uniques,emails_communs,emails_nouveaux_taxe
0,5778,98,68,30


## 7. Présence d'une information "taxe" dans la feuille 1 (contacts)

Objectif : vérifier si les adresses e-mail de la feuille 2 (taxe) sont présentes dans la feuille 1 (contacts).

In [44]:
# Analyse de la colonne 'Assujetti.Entité' dans la feuille 1
df_contacts["Assujetti.Entité"].value_counts(dropna=False)


Assujetti.Entité
1.0    6468
NaN     331
Name: count, dtype: int64

## 8. Génération du JSONL intermédiaire — Feuille 2 : Taxe

Objectif : générer une structure JSONL intermédiaire à partir des données analysées, en vue de leur intégration dans MongoDB et de leur affichage dans le CRM.

In [48]:
# =========================================================
# Génération du JSONL intermédiaire – Feuille 2 : Taxe
# =========================================================

# Construction de la structure intermédiaire
records_taxe = []

for email in sorted(emails_taxe.unique()):
    record = {
        "email": email,
        "taxe": {
            "annee": 2025,
            "verseur": True,
            "present_dans_feuille_1": email in emails_f1
        },
        "source": "feuille_2_taxe"
    }
    records_taxe.append(record)

# Conversion en DataFrame (facilite l'export JSONL)
df_taxe_json = pd.DataFrame(records_taxe)

# Export au format JSONL
output_path = "taxe_2025_intermediaire.jsonl"
df_taxe_json.to_json(output_path, orient="records", lines=True, force_ascii=False)

output_path, df_taxe_json.head()


('taxe_2025_intermediaire.jsonl',
                                 email  \
 0                  a.combier@arcds.fr   
 1       admin@dmgmori-sailingteam.com   
 2  alexandra.lafrogne@naval-group.com   
 3          amani.jebali@capgemini.com   
 4     amaury.le-hir-reynaud@arkea.com   
 
                                                 taxe          source  
 0  {'annee': 2025, 'verseur': True, 'present_dans...  feuille_2_taxe  
 1  {'annee': 2025, 'verseur': True, 'present_dans...  feuille_2_taxe  
 2  {'annee': 2025, 'verseur': True, 'present_dans...  feuille_2_taxe  
 3  {'annee': 2025, 'verseur': True, 'present_dans...  feuille_2_taxe  
 4  {'annee': 2025, 'verseur': True, 'present_dans...  feuille_2_taxe  )

## 9. Conclusion – Feuille 2 : Taxe

La feuille « Taxe » contient une liste de 101 adresses e-mail associées aux verseurs de la taxe d’apprentissage 2025.
Après normalisation minimale, 98 adresses sont uniques, avec 3 doublons (taux de doublons ≈ 2,97 %).

Le recouvrement avec la feuille 1 montre que cette feuille sert majoritairement à enrichir la base principale : 68 adresses e-mail sont déjà présentes parmi les contacts de la feuille 1, tandis que 30 adresses sont nouvelles et peuvent être ajoutées pour compléter la base principale des contacts.
Cette feuille ne représente donc pas une base de contacts autonome, mais une source d’enrichissement de la base existante.

Enfin, la feuille 1 contient déjà une information liée à la taxe via la colonne `Assujetti.Entité` mais sans précision temporelle. La feuille « Taxe » apporte une liste explicite et datée (année 2025) pouvant être utilisée comme source métier complémentaire.


Dans ce cadre, un JSONL intermédiaire a été généré pour la feuille 2. Il permet d’associer, pour chaque adresse e-mail, l’information « verseur de taxe 2025 » ainsi que son statut de présence ou non dans la feuille 1. Ce format est destiné à une intégration progressive dans Mongo/CremeCRM, en complément des contacts issus de la feuille 1.



# EDA – Feuille 3 : Pour e-mailing

### Objectif
Analyser la feuille « Pour e-mailing » afin de :
- vérifier la structure et la qualité des adresses emails,
- identifier les doublons,
- mesurer le recouvrement avec les feuille 1 et 2,
- évaluer l'exploitabilité de cette feuille pour des actions d'emailing,
- générer un JSONL intermédiaire.

## 1. Chargement et structure

Objectif : charger la feuille 3 "Pour e-mailing" et vérifier sa structure (nombre de lignes, de colonnes, nom des colonnes).

In [49]:
# # Chargement et structure de la feuille 3 : "Pour e-mailing"
df_emailing = pd.read_excel(
    file_path,
    sheet_name="Pour e-mailing"
)

df_emailing.shape, df_emailing.columns


((5563, 1), Index(['Coordonnee.Coordonnee'], dtype='str'))

## 2. Aperçu du contenu

Objectif : vérifier qu'il s'agit bien uniquement d'adresses e-mails

In [50]:
# Aperçu des premières lignes
df_emailing.head(10)


,Coordonnee.Coordonnee
0,1095.1096.TEAM.ASSISTANTES@soprasteria.com
1,123hr@axa-im.com
2,a.abedin@nexter-group.fr
3,a.blandel@lacroix.group
4,a.blin@leongrosse.fr
5,a.bouttier@fr.merce.mee.com
6,a.chagnon@accenture.com
7,a.chatelain@clemessy.fr
8,a.clement@morbihan-auto.com
9,a.combier@arcds.fr


## 3. Valeurs manquantes

Objectif : mesurer la qualité des données par colonne afin d’identifier les champs critiques.

In [51]:
# Analyse des valeurs manquantes
missing_count = df_emailing.isna().sum()
missing_percent = (df_emailing.isna().mean() * 100).round(2)

pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})


,missing_count,missing_percent
Coordonnee.Coordonnee,0,0.0


## 4. Normalisation minimale des adresses e-mail

Objectif : supprimer les valeurs manquantes, convertir en chaine de caractères, supprimer les espaces en début et fin de chaîne, passer en minuscules pour obtenir une comparaison fiable des adresses e-mail.

In [52]:
# =========================================================
# Étape 2 : normalisation minimale
# =========================================================

emails_emailing = (
    df_emailing["Coordonnee.Coordonnee"]
    .dropna()     # suppression des valeurs manquantes
    .astype(str)  # conversion en texte (éviter les erreurs liées aux NaN)
    .str.strip()  # suppression des espaces en début / fin
    .str.lower()  # mise en minuscules (pour comparaison fiable)
)

emails_emailing.head(10)


0    1095.1096.team.assistantes@soprasteria.com
1                              123hr@axa-im.com
2                      a.abedin@nexter-group.fr
3                       a.blandel@lacroix.group
4                          a.blin@leongrosse.fr
5                   a.bouttier@fr.merce.mee.com
6                       a.chagnon@accenture.com
7                       a.chatelain@clemessy.fr
8                   a.clement@morbihan-auto.com
9                            a.combier@arcds.fr
Name: Coordonnee.Coordonnee, dtype: str

## 5. Analyse des doublons d’adresses e-mail

Objectif : identifier la présence éventuelle de doublons d'adresses email.

In [53]:
# identification des doublons
total_emails = len(emails_emailing)  # nombre total d'emails
unique_emails = emails_emailing.nunique() # nombre d'emails distincts
duplicate_count = total_emails - unique_emails # nombre d'emails dupliqués
duplicate_rate = round((duplicate_count / total_emails) * 100, 2) # taux de doublons

pd.DataFrame([{
    "emails_total": total_emails,
    "emails_uniques": unique_emails,
    "doublons": duplicate_count,
    "taux_doublons_%": duplicate_rate
}])


,emails_total,emails_uniques,doublons,taux_doublons_%
0,5563,5561,2,0.04


## 6. Contrôles simple de qualité des adresses e-mail

Objectif : évaluer la qualité des adresses e-mail (format, valeurs aberrantes) afin d’anticiper les règles de validation et de nettoyage lors de la phase ETL.

In [54]:
# liste des emails dupliqués (pour contrôle)
empty_after_strip = (emails_emailing.str.len() == 0).sum()
contains_inner_spaces = emails_emailing.str.contains(r"\s", na=False).sum()
multiple_at = (emails_emailing.str.count("@") > 1).sum()
no_dot_after_at = emails_emailing[
    ~emails_emailing.str.split("@").str[1].str.contains(r"\.", na=False)
].shape[0]

pd.DataFrame([
    {"check": "vides après strip", "count": int(empty_after_strip), "percent": round(empty_after_strip/total_emails*100, 2)},
    {"check": "contient des espaces", "count": int(contains_inner_spaces), "percent": round(contains_inner_spaces/total_emails*100, 2)},
    {"check": "plusieurs @", "count": int(multiple_at), "percent": round(multiple_at/total_emails*100, 2)},
    {"check": "pas de '.' après @", "count": int(no_dot_after_at), "percent": round(no_dot_after_at/total_emails*100, 2)},
])


,check,count,percent
0,vides après strip,0,0.00
1,contient des espaces,0,0.00
2,plusieurs @,4,0.07
3,pas de '.' après @,7,0.13


In [ ]:
# Exemples d'anomalies possibles (doubles espaces, plusieurs @, pas de . dans le domaine)
emails_emailing[emails_emailing.str.contains(r"\s", na=False)].head(10), \ 
emails_emailing[emails_emailing.str.count("@") > 1].head(10), \
emails_emailing[
    ~emails_emailing.str.split("@").str[1].str.contains(r"\.", na=False)
].head(10)


(Series([], Name: Coordonnee.Coordonnee, dtype: str),
 242                     alpenfor@bouyguestelecom@.fr
 3334    lucie.leffray@cf.grouplucie.leffray@cf.group
 3608                    mathieu@collignon@airbus.com
 3736                  mhuchet@@mobilitytechgreen.com
 Name: Coordonnee.Coordonnee, dtype: str,
 242       alpenfor@bouyguestelecom@.fr
 444               antoine.vignonvif.fr
 3608      mathieu@collignon@airbus.com
 3646                  matthieu@evernet
 3736    mhuchet@@mobilitytechgreen.com
 3952            nicolas.cariou@gwagenn
 4265        pdv044006mousquetaires.com
 Name: Coordonnee.Coordonnee, dtype: str)

## 7. Recouvrement avec les feuilles 1 et 2

Objectif : mesurer le recouvrement des adresses e-mail entre la feuille 3 (pour e-mailing) et les feuilles 1 et 2 afin d’évaluer comment cette feuille peut enrichir les données existantes.

In [56]:
# Recouvrement avec les emails de la feuille 1
emails_communs_f1 = set(emails_emailing) & set(emails_f1)

pd.DataFrame([{
    "emails_emailing_uniques": unique_emails,
    "emails_communs_avec_feuille_1": len(emails_communs_f1),
    "emails_absents_feuille_1": unique_emails - len(emails_communs_f1)
}])


,emails_emailing_uniques,emails_communs_avec_feuille_1,emails_absents_feuille_1
0,5561,5514,47


In [57]:
# Recouvrement avec les emails de la feuille 2
emails_communs_f2 = set(emails_emailing) & set(emails_taxe)

pd.DataFrame([{
    "emails_communs_avec_feuille_2": len(emails_communs_f2)
}])


,emails_communs_avec_feuille_2
0,93


## 8. Génération du JSONL intermédiaire — Feuille 3 : Pour e-mailing

Objectif : générer une structure JSONL intermédiaire à partir des données analysées, en vue de leur intégration dans MongoDB et de leur affichage dans le CRM.

In [ ]:
# =========================================================
# Génération du JSONL intermédiaire – Feuille 3 : Pour e-mailing
# =========================================================

# Initialisation de la liste des enregistrements
records_emailing = []

# Construction de la structure intermédiaire
for email in sorted(emails_emailing.unique()):
    record = {
        "email": email,
        "emailing": {
            "source": "feuille_3_pour_emailing",
            "present_dans_feuille_1": email in emails_f1,
            "present_dans_feuille_2_taxe": email in emails_taxe
        }
    }
    records_emailing.append(record)

# Conversion en DataFrame pour export
df_emailing_json = pd.DataFrame(records_emailing)

# Export au format JSONL
output_path = "emailing_intermediaire.jsonl"
df_emailing_json.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# Résultat
output_path, df_emailing_json.head()


('emailing_intermediaire.jsonl',
                                         email  \
 0  1095.1096.team.assistantes@soprasteria.com   
 1                            123hr@axa-im.com   
 2                           a-l-f2@wanadoo.fr   
 3                    a.abedin@nexter-group.fr   
 4                     a.blandel@lacroix.group   
 
                                             emailing  
 0  {'source': 'feuille_3_pour_emailing', 'present...  
 1  {'source': 'feuille_3_pour_emailing', 'present...  
 2  {'source': 'feuille_3_pour_emailing', 'present...  
 3  {'source': 'feuille_3_pour_emailing', 'present...  
 4  {'source': 'feuille_3_pour_emailing', 'present...  )

In [59]:
# Affichage des 2 premières lignes du fichier JSONL généré pour vérification
with open("emailing_intermediaire.jsonl", encoding="utf-8") as f:
    for _ in range(2):
        print(next(f))


{"email":"1095.1096.team.assistantes@soprasteria.com","emailing":{"source":"feuille_3_pour_emailing","present_dans_feuille_1":true,"present_dans_feuille_2_taxe":false}}

{"email":"123hr@axa-im.com","emailing":{"source":"feuille_3_pour_emailing","present_dans_feuille_1":true,"present_dans_feuille_2_taxe":false}}



## 9. Conclusion – Feuille 3 : Pour e-mailing

La feuille « Pour e-mailing » contient une base d’adresses e-mail destinée à des actions de communication. Les données sont structurées autour d’une seule colonne ("Coordonnee.Coordonnee") et présentent une volumétrie plus importante que les feuilles précédentes : 5563 adresses e-mail, dont 5561 uniques, avec un taux de doublons très faible  (2 doublons, soit 0,036%).

Les contrôles simples de qualité montrent une très bonne homogénéité des données. Quelques anomalies de format sont détectées (présence de plusieurs caractères `@` ou absence de `.` dans le domaine), mais elles restent marginales et concernent un nombre très limité de lignes.


Le recouvrement avec la feuille 1 est très élevé :

5514 adresses e-mail sont déjà présentes dans la base principle de contacts, tandis que 47 adresses ne le lien avec la feuille 2 montre également un recouvrement significatif, avec 93 adresses e-mail présentes dans les deux bases.

Cette feuille constitue donc une base solide pour des actions d’e-mailing, principalement en appui de la base de contacts existante, sous réserve d'une mise en cohérence avec les r-gles de consentement et les statuts de refus identifiés dans les autres sources.

Un JSONL intermédiaire a été généré à l’issue de cette analyse.
Ce fichier associe à chaque adresse e-mail des indicateurs de présence dans la feuille 1 et dans la feuille 2, ainsi que l’identification de la source. Il constitue une sortie intermédiaire destinée à une intégration progressive dans Mongo/CremeCRM, en cohérence avec les autres JSONL produits à partir de ce fichier.


# Consolidation – JSONL global


Objectif : fusionner les JSONL intermédiaires des feuilles 1, 2 et 3 en un JSONL global, en utilisant l’adresse e-mail comme clé de consolidation.


In [62]:
# =========================================================
# Consolidation – Génération d'un JSONL global
# Clé de consolidation : adresse e-mail
# =========================================================

import pandas as pd

# --- Chemins des fichiers JSONL intermédiaires ---
# Les fichiers sont générés à l'issue des EDA des feuilles 1, 2 et 3
path_f1 = "contacts_intermediaire.jsonl"
path_f2 = "taxe_2025_intermediaire.jsonl"
path_f3 = "emailing_intermediaire.jsonl"

# --- Lecture des fichiers JSONL ---
df_f1 = pd.read_json(path_f1, lines=True)
df_f2 = pd.read_json(path_f2, lines=True)
df_f3 = pd.read_json(path_f3, lines=True)

# --- Extraction de l'adresse e-mail depuis la structure "coordonnees" ---
# Dans le JSONL issu de la feuille 1, les coordonnées sont stockées
# sous forme de dictionnaire. Lorsque l'information est disponible,
# l'adresse e-mail est portée par la clé "email".
# Cette étape crée une colonne "email" utilisée comme clé de consolidation.
df_f1["email"] = df_f1["coordonnees"].apply(
    lambda x: x.get("email") if isinstance(x, dict) else None
)

# --- Filtrage : conservation uniquement des lignes disposant d'une adresse e-mail ---
# Les lignes sans email ne peuvent pas être consolidées avec les autres sources.
df_f1_emails = df_f1.dropna(subset=["email"]).copy()

# --- Préparation des données issues des feuilles 2 et 3 ---
# Ces fichiers portent déjà une colonne "email" et ne nécessitent
# pas de transformation supplémentaire à ce stade.
df_f2_emails = df_f2.copy()
df_f3_emails = df_f3.copy()

# --- Fusion des sources ---
# Les fusions sont réalisées en jointure externe (outer join)
# afin de conserver l'ensemble des adresses e-mail présentes
# dans au moins une des sources.
df_global = df_f1_emails.merge(
    df_f2_emails,
    on="email",
    how="outer",
    suffixes=("", "_taxe")
)

df_global = df_global.merge(
    df_f3_emails,
    on="email",
    how="outer",
    suffixes=("", "_emailing")
)

# --- Export du JSONL global ---
# Chaque ligne du fichier correspond à une adresse e-mail consolidée,
# regroupant les informations issues des différentes feuilles.
output_global = "contacts_global.jsonl"
df_global.to_json(
    output_global,
    orient="records",
    lines=True,
    force_ascii=False
)

# --- Résumé ---
output_global, df_global.shape, df_global.head(3)


('contacts_global.jsonl',
 (6519, 9),
                                               entite  \
 0  {'code': 'DEVOTEAM_44', 'libelle': 'DEVOTEAM',...   
 1  {'code': 'GEMO_DIEPPE_76', 'libelle': 'GEMO', ...   
 2  {'code': 'SOPRASTERIA_44_SAINT_HERBLAIN_REGGIA...   
 
                                              adresse  \
 0  {'ligne1': '9 bis rue Emile Masson', 'compleme...   
 1  {'ligne1': 'Cc Le Belvedere', 'complement': 'A...   
 2  {'ligne1': 'Batiment Ar Mor II', 'complement':...   
 
                                              contact  \
 0  {'civilite': 'Madame', 'nom': 'GOMBERT', 'pren...   
 1  {'civilite': 'Madame', 'nom': 'DEVACHT', 'pren...   
 2  {'civilite': 'Madame', 'nom': 'DIDIER', 'preno...   
 
                                          coordonnees consent  \
 0  {'email': '.gombert@devoteam.com', 'telephone'...    None   
 1  {'email': '05610@pdv.gemo.fr', 'telephone': No...    None   
 2  {'email': '1095.1096.team.assistantes@sopraste...    None   
 
          

# Conclusion globale – Fichier

"CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026"


L’analyse exploratoire de ce fichier met en évidence une base riche mais hétérogène, composée de trois feuilles répondant à des objectifs distincts : constitution de la base de contacts, identification des verseurs de la taxe d’apprentissage pour 2025 et préparation d’actions d’e-mailing. 

La feuille 1 constitue le socle principal du fichier. Elle regroupe les informations relatives aux entités, aux individus et à leurs coordonnées. L’EDA a montré que certaines colonnes sont très largement renseignées (identité des entités, localisation, nom et prénom des contacts), tandis que d’autres présentent des taux de complétude plus faibles ou des usages multiples. En particulier, la colonne Coordonnée.Coordonnée agrège plusieurs types d’informations (adresses e-mail, numéros de téléphone, statuts de refus ou de désinscription), parfois concaténées, ce qui rend nécessaire une structuration préalable avant toute exploitation CRM. Un JSONL intermédiaire a été généré à l’issue de cette analyse afin de représenter les contacts de manière structurée, en séparant clairement les coordonnées et les informations de consentement, tout en conservant la valeur brute pour la traçabilité.

La feuille 2 (Taxe) apporte une information métier complémentaire. Elle contient une liste ciblée d’adresses e-mail associées aux verseurs de la taxe d’apprentissage pour l’année 2025. L’analyse a mis en évidence un faible taux de doublons et un recouvrement important avec la feuille 1, confirmant que cette feuille ne constitue pas une base de contacts autonome, mais un enrichissement de la base existante. Un JSONL intermédiaire dédié a été produit afin d’associer explicitement à chaque adresse e-mail l’information "verseur de taxe 2025" et son lien éventuel avec les contacts déjà présents.

La feuille 3 (Pour e-mailing) correspond à une base d’adresses e-mail destinée à des actions de communication. Les contrôles de qualité montrent une bonne homogénéité des données, avec des anomalies de format très marginales. Le recouvrement avec les feuilles 1 et 2 est élevé, indiquant que cette feuille sert principalement d’appui aux contacts existants, tout en faisant apparaître un nombre limité d’adresses à qualifier. Un JSONL intermédiaire a également été généré pour cette feuille, afin de conserver l’information de présence dans la base e-mailing et les liens avec les autres sources.

À partir de ces trois JSONL intermédiaires, un JSONL global a été construit. Il consolide l’ensemble des informations disponibles autour de la clé « adresse e-mail », en regroupant les données issues des trois feuilles sans fusion prématurée ni perte de traçabilité. Ce fichier global constitue une vue cohérente et exploitable de l’état des données à un instant donné.

En conclusion, ce travail d’EDA a permis de :

- comprendre la structure et les usages des différentes feuilles,
- identifier les points de vigilance en termes de qualité et de structuration des données,
- produire des sorties intermédiaires JSONL prêtes pour une intégration progressive dans Mongo/CremeCRM.

Les prochaines étapes consisteront à intégrer ces données dans la base, à poursuivre l’analyse sur les autres fichiers reçus et à affiner la structuration en fonction des besoins fonctionnels du CRM, sans remettre en cause les choix posés à ce stade.

